# Paired-End RNA-seq Pipeline

**Workflow:** SRA download → quality trimming (cutadapt) → alignment (HISAT2) → read counting (featureCounts)

**Dependencies:** `sratoolkit`, `cutadapt`, `hisat2`, `subread` (featureCounts)

---

## 0. Configuration

Edit the variables in this cell before running the rest of the notebook.

In [ ]:
# ── SRA accessions to process ─────────────────────────────────────────────
SRR_IDS = [
    "SRR000001",
    "SRR000002",
    "SRR000003",
    "SRR000004",
    "SRR000005",
    "SRR000006",
]

# Maximum reads to download per accession (set to None to download all reads)
MAX_READS = 5_000_000

# HISAT2 genome index prefix (e.g. 'mm10/genome' or 'hg38/genome')
HISAT2_INDEX = "mm10/genome"

# Ensembl GTF annotation file
GTF = "Mus_musculus.GRCm38.102.gtf"

# Alignment threads
THREADS = 4

## 1. Install SRA Toolkit

In [ ]:
!wget -q https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/3.1.0/sratoolkit.3.1.0-ubuntu64.tar.gz
!tar -xzf sratoolkit.3.1.0-ubuntu64.tar.gz
!chmod +x sratoolkit.3.1.0-ubuntu64/bin/fastq-dump

import os
os.environ["PATH"] += ":" + os.path.abspath("sratoolkit.3.1.0-ubuntu64/bin")

## 2. Install HISAT2 Genome Index

Replace the URL below with the appropriate pre-built index for your organism/assembly.
Pre-built indexes are available at https://daehwankimlab.github.io/hisat2/download/

In [ ]:
!wget -q https://genome-idx.s3.amazonaws.com/hisat/mm10_genome.tar.gz
!tar -xzf mm10_genome.tar.gz

## 3. Install cutadapt and featureCounts (subread)

In [ ]:
!pip install -q cutadapt
!apt-get install -qq hisat2 subread

## 4. Download GTF Annotation

Replace the URL below with the GTF matching your genome assembly.
GTF files are available at https://ftp.ensembl.org/pub/

In [ ]:
!wget -q https://ftp.ensembl.org/pub/release-102/gtf/mus_musculus/Mus_musculus.GRCm38.102.gtf.gz
!gunzip -f Mus_musculus.GRCm38.102.gtf.gz

## 5. Download Paired-End FASTQ Files

Downloads forward (`_1`) and reverse (`_2`) reads for each accession.
The `--split-files` flag separates paired reads into distinct files.

In [ ]:
import os

for srr in SRR_IDS:
    if os.path.exists(f"{srr}_1.fastq") and os.path.exists(f"{srr}_2.fastq"):
        print(f"Skipping {srr} — FASTQ files already exist.")
        continue
    print(f"Downloading {srr}...")
    limit_flag = f"-X {MAX_READS}" if MAX_READS else ""
    !fastq-dump --split-files {limit_flag} {srr}

## 6. Quality Trimming (cutadapt)

- `-q 20` — trims low-quality bases (Phred < 20) from the 3' end  
- `--minimum-length 30` — discards reads shorter than 30 bp after trimming

In [ ]:
import os

for srr in SRR_IDS:
    r1_trim = f"{srr}_1_trimmed.fastq"
    r2_trim = f"{srr}_2_trimmed.fastq"

    if os.path.exists(r1_trim) and os.path.exists(r2_trim):
        print(f"Skipping {srr} — trimmed files already exist.")
        continue

    print(f"Trimming {srr}...")
    !cutadapt \
        -q 20 \
        --minimum-length 30 \
        -o {srr}_1_trimmed.fastq \
        -p {srr}_2_trimmed.fastq \
        {srr}_1.fastq {srr}_2.fastq

## 7. Alignment to Reference Genome (HISAT2)

Aligns trimmed paired-end reads to the reference genome and writes output as SAM files.

In [ ]:
import os

for srr in SRR_IDS:
    sam = f"{srr}.sam"

    if os.path.exists(sam):
        print(f"Skipping {srr} — SAM file already exists.")
        continue

    print(f"Aligning {srr}...")
    !hisat2 \
        -p {THREADS} \
        -x {HISAT2_INDEX} \
        -1 {srr}_1_trimmed.fastq \
        -2 {srr}_2_trimmed.fastq \
        -S {sam}

## 8. Generate Counts Table (featureCounts)

Counts the number of read pairs mapping to each gene across all samples.

- `-p` — paired-end mode  
- `--countReadPairs` — counts fragments (read pairs) rather than individual reads  

**Output:** `counts.txt` — import this file into R for DEG analysis (DESeq2 / edgeR).

In [ ]:
sam_files = " ".join([f"{srr}.sam" for srr in SRR_IDS])

!featureCounts \
    -p \
    --countReadPairs \
    -T {THREADS} \
    -a {GTF} \
    -o counts.txt \
    {sam_files}

## 9. Inspect Output

Preview the counts table. Download `counts.txt` for downstream DEG analysis in R (DESeq2 / edgeR).

In [ ]:
!head -3 counts.txt

In [ ]:
!cat counts.txt.summary